# Precompute Reference Mapping — Helical Bio Explorer

**Author:** (your name)  
**Date:** (run date)

This notebook runs the **offline** reference-mapping pipeline for the demo: load PBMC 3k (healthy reference) and childhood cSLE PBMCs (GSE135779), compute **Geneformer** and **GenePT** embeddings via the [Helical](https://github.com/helicalAI/helical) SDK, fit **UMAP** on healthy cells, project disease cells, score distances to healthy centroids, export **parquet** artifacts to **S3**, and write **provenance** rows to Postgres.

**Expected runtime:** about **30–60 minutes** on a Colab **T4 GPU**, depending on network (GEO download) and whether you limit the number of 10x libraries via `CSLE_MAX_SAMPLES`.

See `notebooks/README.md` for environment variables and troubleshooting.


In [ ]:
# Pinned dependencies (upload `requirements-colab.txt` next to this notebook, or run from repo root).
# From repo root: `notebooks/requirements-colab.txt` exists. From Colab-only upload: use `requirements-colab.txt`.
import subprocess
from pathlib import Path

_req = Path("notebooks/requirements-colab.txt")
if not _req.is_file():
    _req = Path("requirements-colab.txt")
if not _req.is_file():
    msg = "Missing requirements-colab.txt — clone the repo or upload notebooks/requirements-colab.txt"
    raise FileNotFoundError(msg)
subprocess.check_call(["pip", "install", "-r", str(_req)])


In [ ]:
# Configuration: secrets, AWS, S3 version prefix, git SHA (provenance).
from __future__ import annotations

import os
import subprocess
from pathlib import Path


def _secret(name: str) -> str | None:
    v = os.environ.get(name)
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore[import-not-found]

        return userdata.get(name)
    except Exception:
        return None


DATABASE_URL = _secret("DATABASE_URL") or ""
S3_BUCKET = _secret("S3_BUCKET") or ""
S3_REGION = _secret("S3_REGION") or "us-east-1"
AWS_ACCESS_KEY_ID = _secret("AWS_ACCESS_KEY_ID") or ""
AWS_SECRET_ACCESS_KEY = _secret("AWS_SECRET_ACCESS_KEY") or ""

os.environ.setdefault("AWS_ACCESS_KEY_ID", AWS_ACCESS_KEY_ID)
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", AWS_SECRET_ACCESS_KEY)
os.environ.setdefault("AWS_DEFAULT_REGION", S3_REGION)

VERSION = (os.environ.get("PRECOMPUTE_RUN_VERSION") or "1").strip()
GIT_SHA = "unknown"
try:
    GIT_SHA = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        text=True,
        stderr=subprocess.DEVNULL,
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_SHA = (os.environ.get("GIT_SHA") or "unknown").strip()

print("Config summary (non-secret):")
print(f"  VERSION (S3 prefix): v{VERSION}")
print(f"  GIT_SHA: {GIT_SHA[:12]}…" if len(GIT_SHA) > 12 else f"  GIT_SHA: {GIT_SHA}")
print(f"  S3_BUCKET set: {bool(S3_BUCKET)}")
print(f"  S3_REGION: {S3_REGION}")
print(f"  DATABASE_URL set: {bool(DATABASE_URL)}")
print(f"  AWS_ACCESS_KEY_ID set: {bool(AWS_ACCESS_KEY_ID)}")


## Step 1: Load datasets

Healthy reference: **PBMC 3k** (`scanpy.datasets.pbmc3k_processed`). Disease query: **GSE135779** childhood cSLE cohort — expression from GEO supplementary `GSE135779_RAW.tar`, metadata (SLEDAI, labels) from the author GitHub CSV linked in `notebooks/README.md`.


In [ ]:
# PBMC 3k (healthy reference) — scanpy built-in, already cell-type labeled (Louvain clusters).
import scanpy as sc

adata_healthy = sc.datasets.pbmc3k_processed()
assert adata_healthy.n_obs > 0 and adata_healthy.n_vars > 0

adata_healthy.obs["cell_type"] = adata_healthy.obs["louvain"].astype(str)

print("PBMC 3k shape:", adata_healthy.shape)
print(adata_healthy.obs["cell_type"].value_counts())


In [ ]:
# cSLE disease cohort (GSE135779): download GEO supplementary tar + author metadata, build AnnData, harmonize labels.
import os
import re
import shutil
import tarfile
import tempfile
from pathlib import Path

import GEOparse
import pandas as pd
from anndata import AnnData, concat as ann_concat
import requests
import scanpy as sc
from tqdm.auto import tqdm

CSLE_META_URL = (
    "https://raw.githubusercontent.com/dnehar/SingleCells_SLE_paper/master/"
    "Meta_cSLE_processed_0809202_small.csv"
)
GEO_TAR_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE135nnn/GSE135779/suppl/GSE135779_RAW.tar"
)


def harmonize_cell_type(subcluster: str) -> str:
    """Map author subcluster codes to PBMC 3k-style coarse labels."""
    s = str(subcluster)
    if s.startswith("B-") or s.startswith("B_SC"):
        return "B cells"
    if s.startswith("Mono") or "Mono" in s:
        return "CD14+ Monocytes"
    if s.startswith("NK"):
        return "NK cells"
    if "DC" in s:
        return "Dendritic cells"
    if "Mega" in s or "Platelet" in s:
        return "Megakaryocytes"
    if "FCGR3A" in s or "CD16" in s:
        return "FCGR3A+ Monocytes"
    if s.startswith("T-"):
        return "CD4 T cells"
    return "CD4 T cells"


def disease_activity_from_meta(row: pd.Series) -> str:
    """SLEDAI: low <= 4, high > 4 using author `SLEDAI_cat` when present."""
    cat = str(row.get("SLEDAI_cat", "")).lower()
    if "<=4" in cat or cat.strip() == "low":
        return "low"
    if ">4" in cat or "high" in cat:
        return "high"
    # Fallback: parse numeric ranges in `SLEDAI` like "0 to 3"
    raw = str(row.get("SLEDAI", "")).lower()
    nums = [int(x) for x in re.findall(r"\d+", raw)]
    if not nums:
        return "low"
    hi = max(nums)
    return "low" if hi <= 4 else "high"


def gsm_to_library_id(gsm) -> str:
    title = " ".join(gsm.metadata.get("title", []) or [])
    src = " ".join(gsm.metadata.get("source_name_ch1", []) or [])
    for blob in (title, src):
        m = re.search(r"(JB\d+)", blob)
        if m:
            return m.group(1)
    return gsm.name


try:
    meta = pd.read_csv(CSLE_META_URL)
except Exception as exc:
    raise RuntimeError(
        "Failed to download cSLE metadata CSV. Check network / GitHub availability."
    ) from exc

meta["cell_key"] = meta["IDs"].astype(str) + "_" + meta["index"].astype(str)

try:
    gse = GEOparse.get_GEO("GSE135779", destdir="./geo_soft", silent=True)
    gsm_map = {name: gsm_to_library_id(gsm) for name, gsm in gse.gsms.items()}
except Exception as exc:
    raise RuntimeError(
        "GEOparse failed to load GSE135779 SOFT metadata. Retry or update GEOparse."
    ) from exc

cache_dir = Path("./geo_cache")
cache_dir.mkdir(parents=True, exist_ok=True)
tar_path = Path(os.environ.get("CSLE_RAW_TAR", "")).expanduser()
if str(tar_path) == "." or not tar_path.is_file():
    tar_path = cache_dir / "GSE135779_RAW.tar"
    if not tar_path.is_file():
        try:
            with requests.get(GEO_TAR_URL, stream=True, timeout=60) as r:
                r.raise_for_status()
                total = int(r.headers.get("Content-Length", "0"))
                pbar = tqdm(
                    total=total, unit="B", unit_scale=True, desc="Downloading GSE135779_RAW.tar"
                )
                with open(tar_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
                pbar.close()
        except Exception as exc:
            raise RuntimeError(
                "GEO tar download failed. Retry, check Colab network, or set CSLE_RAW_TAR "
                "to a local copy of GSE135779_RAW.tar."
            ) from exc

extract_root = cache_dir / "GSE135779_RAW_extracted"
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

with tarfile.open(tar_path, "r") as tf:
    tf.extractall(path=extract_root)

max_samples = os.environ.get("CSLE_MAX_SAMPLES")
max_n = int(max_samples) if max_samples else None

# Collect 10x matrix directories: each GSM prefix should have matrix/features/barcodes.
matrix_files = sorted(extract_root.rglob("*matrix.mtx.gz"))
if not matrix_files:
    raise RuntimeError("No *matrix.mtx.gz found after extracting GSE135779_RAW.tar")

adatas: list[AnnData] = []
for mtx_path in matrix_files:
    if max_n is not None and len(adatas) >= max_n:
        break
    sample_dir = mtx_path.parent
    m = re.search(r"(GSM\d+)", mtx_path.name)
    gsm = m.group(1) if m else mtx_path.name
    lib = gsm_map.get(gsm, gsm)
    try:
        a = sc.read_10x_mtx(sample_dir, var_names="gene_symbols", cache=False)
    except Exception:
        continue
    a.obs_names = [f"{lib}_{bc}" for bc in a.obs_names]
    adatas.append(a)

if not adatas:
    raise RuntimeError("Could not read any 10x matrices from extracted GEO tar.")

adata_all = ann_concat(adatas, merge="same", axis=0)
adata_all.obs["cell_key"] = adata_all.obs_names
adata_all.obs = adata_all.obs.join(meta.set_index("cell_key"), how="inner")

# Disease query cells: childhood SLE patients only (exclude healthy controls in this cohort).
adata_disease = adata_all[adata_all.obs["Groups"] == "cSLE"].copy()
adata_disease.obs["cell_type"] = [harmonize_cell_type(x) for x in adata_disease.obs["subclusters"]]
adata_disease.obs["disease_activity"] = adata_disease.obs.apply(
    disease_activity_from_meta, axis=1
)

print("cSLE AnnData shape (disease only):", adata_disease.shape)
print(pd.crosstab(adata_disease.obs["cell_type"], adata_disease.obs["disease_activity"]))


## Step 2: Geneformer embeddings

We use the **v1-scale** checkpoint `gf-6L-10M-i2048` (2048 ranked genes per cell) with **cell** embedding mode. The same model instance embeds both reference and disease for consistent geometry.


In [ ]:
# Geneformer (healthy + disease) — Helical SDK.
import numpy as np
import torch
from helical.models.geneformer import Geneformer, GeneformerConfig

_device = "cuda" if torch.cuda.is_available() else "cpu"
_gf_cfg = GeneformerConfig(
    model_name="gf-6L-10M-i2048",
    batch_size=16,
    emb_mode="cell",
    device=_device,  # Colab T4 uses CUDA when available
    custom_attr_name_dict={"cell_type": "cell_type"},
)
geneformer = Geneformer(_gf_cfg)

gf_h = geneformer.process_data(adata_healthy)
emb_h = geneformer.get_embeddings(gf_h)
adata_healthy.obsm["X_geneformer"] = np.asarray(emb_h, dtype=np.float32)
print("Geneformer healthy embedding shape:", adata_healthy.obsm["X_geneformer"].shape)


In [ ]:
gf_d = geneformer.process_data(adata_disease)
emb_d = geneformer.get_embeddings(gf_d)
adata_disease.obsm["X_geneformer"] = np.asarray(emb_d, dtype=np.float32)
print("Geneformer disease embedding shape:", adata_disease.obsm["X_geneformer"].shape)


## Step 3: GenePT embeddings

GenePT uses **text-derived gene embeddings** (here `gpt3.5` backend in Helical) with **cell** mode — a second geometry for cross-model disagreement versus Geneformer.


In [ ]:
from helical.models.genept import GenePT, GenePTConfig

_gpt_cfg = GenePTConfig(
    model_name="gpt3.5",
    batch_size=24,
    emb_mode="cell",
    device=_device,
    custom_attr_name_dict={"cell_type": "cell_type"},
)
genept = GenePT(_gpt_cfg)

gpt_h = genept.process_data(adata_healthy)
emb_gh = genept.get_embeddings(gpt_h)
adata_healthy.obsm["X_genept"] = np.asarray(emb_gh, dtype=np.float32)
print("GenePT healthy embedding shape:", adata_healthy.obsm["X_genept"].shape)


In [ ]:
gpt_d = genept.process_data(adata_disease)
emb_gd = genept.get_embeddings(gpt_d)
adata_disease.obsm["X_genept"] = np.asarray(emb_gd, dtype=np.float32)
print("GenePT disease embedding shape:", adata_disease.obsm["X_genept"].shape)


## Step 4: UMAP + reference manifold

UMAP is fit **only on healthy cells** to define the reference manifold. We keep the fitted models (`umap_geneformer`, `umap_genept`) for `transform()` on disease embeddings in the next step.


In [ ]:
import pandas as pd
from umap import UMAP

X_hf = adata_healthy.obsm["X_geneformer"]
umap_geneformer = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
adata_healthy.obsm["X_umap_geneformer"] = umap_geneformer.fit_transform(X_hf).astype("float32")

X_hg = adata_healthy.obsm["X_genept"]
umap_genept = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
adata_healthy.obsm["X_umap_genept"] = umap_genept.fit_transform(X_hg).astype("float32")

umap_gf_coords = pd.DataFrame(
    adata_healthy.obsm["X_umap_geneformer"],
    columns=["umap_1", "umap_2"],
    index=adata_healthy.obs_names,
)
umap_gf_coords["cell_type"] = adata_healthy.obs["cell_type"].values
centroids_geneformer = (
    umap_gf_coords.groupby("cell_type", observed=True)[["umap_1", "umap_2"]].mean().reset_index()
)

umap_gpt_coords = pd.DataFrame(
    adata_healthy.obsm["X_umap_genept"],
    columns=["umap_1", "umap_2"],
    index=adata_healthy.obs_names,
)
umap_gpt_coords["cell_type"] = adata_healthy.obs["cell_type"].values
centroids_genept = (
    umap_gpt_coords.groupby("cell_type", observed=True)[["umap_1", "umap_2"]].mean().reset_index()
)

print("Geneformer centroids (UMAP space):")
print(centroids_geneformer)
print("GenePT centroids (UMAP space):")
print(centroids_genept)


## Step 5: Disease projection + distance scores

Project disease **high-dimensional embeddings** into the **healthy UMAP** with `transform()`, then measure **2D Euclidean distance** from each disease cell to the healthy centroid of the **same harmonized cell type**.


In [ ]:
X_df = adata_disease.obsm["X_geneformer"]
adata_disease.obsm["X_umap_geneformer"] = umap_geneformer.transform(X_df).astype("float32")
print("Projected disease UMAP (Geneformer) shape:", adata_disease.obsm["X_umap_geneformer"].shape)


In [ ]:
import numpy as np

cgf_ix = centroids_geneformer.set_index("cell_type")
mean_cgf = cgf_ix[["umap_1", "umap_2"]].mean().to_numpy(dtype=np.float32)
centers_gf = cgf_ix.reindex(adata_disease.obs["cell_type"].astype(str)).to_numpy(dtype=np.float32)
if np.isnan(centers_gf).any():
    nan_rows = np.isnan(centers_gf).any(axis=1)
    centers_gf[nan_rows] = mean_cgf
xy_gf = adata_disease.obsm["X_umap_geneformer"]
adata_disease.obs["distance_geneformer"] = np.linalg.norm(xy_gf - centers_gf, axis=1).astype(np.float32)

summary_gf = (
    adata_disease.obs.groupby(["cell_type", "disease_activity"], observed=True)["distance_geneformer"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)
print("Geneformer distance summaries:")
print(summary_gf)


In [ ]:
X_dg = adata_disease.obsm["X_genept"]
adata_disease.obsm["X_umap_genept"] = umap_genept.transform(X_dg).astype("float32")
print("Projected disease UMAP (GenePT) shape:", adata_disease.obsm["X_umap_genept"].shape)

cgp_ix = centroids_genept.set_index("cell_type")
mean_cgp = cgp_ix[["umap_1", "umap_2"]].mean().to_numpy(dtype=np.float32)
centers_gp = cgp_ix.reindex(adata_disease.obs["cell_type"].astype(str)).to_numpy(dtype=np.float32)
if np.isnan(centers_gp).any():
    nan_rows = np.isnan(centers_gp).any(axis=1)
    centers_gp[nan_rows] = mean_cgp
xy_gp = adata_disease.obsm["X_umap_genept"]
adata_disease.obs["distance_genept"] = np.linalg.norm(xy_gp - centers_gp, axis=1).astype(np.float32)

summary_gp = (
    adata_disease.obs.groupby(["cell_type", "disease_activity"], observed=True)["distance_genept"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)
print("GenePT distance summaries:")
print(summary_gp)


## Step 6: Cross-model disagreement

**Disagreement** = |distance_geneformer − distance_genept| in UMAP-based centroid space. Large values highlight cells where the two foundation models disagree about how far the cell sits from the healthy reference for its cell type.


In [ ]:
import matplotlib.pyplot as plt

dgf = adata_disease.obs["distance_geneformer"].to_numpy(dtype=np.float32)
dgp = adata_disease.obs["distance_genept"].to_numpy(dtype=np.float32)
adata_disease.obs["cross_model_disagreement"] = np.abs(dgf - dgp).astype(np.float32)

summ_disc = (
    adata_disease.obs.groupby(["cell_type", "disease_activity"], observed=True)[
        "cross_model_disagreement"
    ]
    .agg(["mean", "std", "median"])
    .reset_index()
)
print("Cross-model disagreement summaries:")
print(summ_disc)

plt.figure(figsize=(8, 4))
plt.hist(adata_disease.obs["cross_model_disagreement"], bins=64, color="#2c7fb8", edgecolor="white")
plt.title("Cross-model disagreement (|Δ distance|)")
plt.xlabel("disagreement")
plt.ylabel("cell count")
plt.tight_layout()
plt.show()


## Step 7: Export to Parquet

Write six tables (healthy embeddings ×2, disease projections ×2, joint distances, disagreement) under `output/parquet/` and upload to `s3://{bucket}/v{VERSION}/...`.


In [ ]:
# Parquet export + S3 upload (boto3).
from __future__ import annotations

import json
from pathlib import Path

import boto3
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

if not S3_BUCKET:
    raise RuntimeError("S3_BUCKET is not set.")

out_root = Path("output/parquet")
for sub in ("pbmc3k", "sle_csle"):
    (out_root / sub).mkdir(parents=True, exist_ok=True)


def _emb_fields(n_cols: int) -> list[pa.Field]:
    return [pa.field(f"embedding_{j}", pa.float32()) for j in range(n_cols)]


def _write_table(schema: pa.Schema, columns: dict[str, object], path: Path) -> int:
    table = pa.table(columns, schema=schema)
    pq.write_table(table, path)
    return table.num_rows


def _upload(local: Path, key: str) -> None:
    client = boto3.client("s3", region_name=S3_REGION)
    try:
        client.upload_file(str(local), S3_BUCKET, key)
    except Exception as exc:
        raise RuntimeError(f"S3 upload failed for s3://{S3_BUCKET}/{key}") from exc


# --- Healthy Geneformer ---
X_eh_gf = np.asarray(adata_healthy.obsm["X_geneformer"], dtype=np.float32)
U_h_gf = np.asarray(adata_healthy.obsm["X_umap_geneformer"], dtype=np.float32)
n_gf = X_eh_gf.shape[1]
sch_h_gf = pa.schema(
    [
        pa.field("cell_id", pa.string()),
        pa.field("cell_type", pa.string()),
        pa.field("umap_1", pa.float32()),
        pa.field("umap_2", pa.float32()),
        *_emb_fields(n_gf),
    ]
)
cols_h_gf = {
    "cell_id": adata_healthy.obs_names.astype(str).tolist(),
    "cell_type": adata_healthy.obs["cell_type"].astype(str).tolist(),
    "umap_1": U_h_gf[:, 0],
    "umap_2": U_h_gf[:, 1],
}
for j in range(n_gf):
    cols_h_gf[f"embedding_{j}"] = X_eh_gf[:, j]

p_h_gf = out_root / "pbmc3k" / "geneformer_embeddings.parquet"
rows_h_gf = _write_table(sch_h_gf, cols_h_gf, p_h_gf)
key_h_gf = f"v{VERSION}/pbmc3k/geneformer_embeddings.parquet"
_upload(p_h_gf, key_h_gf)

# --- Healthy GenePT ---
X_eh_gp = np.asarray(adata_healthy.obsm["X_genept"], dtype=np.float32)
U_h_gp = np.asarray(adata_healthy.obsm["X_umap_genept"], dtype=np.float32)
n_gp = X_eh_gp.shape[1]
sch_h_gp = pa.schema(
    [
        pa.field("cell_id", pa.string()),
        pa.field("cell_type", pa.string()),
        pa.field("umap_1", pa.float32()),
        pa.field("umap_2", pa.float32()),
        *_emb_fields(n_gp),
    ]
)
cols_h_gp = {
    "cell_id": adata_healthy.obs_names.astype(str).tolist(),
    "cell_type": adata_healthy.obs["cell_type"].astype(str).tolist(),
    "umap_1": U_h_gp[:, 0],
    "umap_2": U_h_gp[:, 1],
}
for j in range(n_gp):
    cols_h_gp[f"embedding_{j}"] = X_eh_gp[:, j]

p_h_gp = out_root / "pbmc3k" / "genept_embeddings.parquet"
rows_h_gp = _write_table(sch_h_gp, cols_h_gp, p_h_gp)
key_h_gp = f"v{VERSION}/pbmc3k/genept_embeddings.parquet"
_upload(p_h_gp, key_h_gp)

# --- Disease projections ---
X_dfull = np.asarray(adata_disease.obsm["X_geneformer"], dtype=np.float32)
X_dgpt = np.asarray(adata_disease.obsm["X_genept"], dtype=np.float32)
U_d_gf = np.asarray(adata_disease.obsm["X_umap_geneformer"], dtype=np.float32)
U_d_gp = np.asarray(adata_disease.obsm["X_umap_genept"], dtype=np.float32)
dist_gf = np.asarray(adata_disease.obs["distance_geneformer"], dtype=np.float32)
dist_gp = np.asarray(adata_disease.obs["distance_genept"], dtype=np.float32)
disag = np.asarray(adata_disease.obs["cross_model_disagreement"], dtype=np.float32)

sch_d_gf = pa.schema(
    [
        pa.field("cell_id", pa.string()),
        pa.field("cell_type", pa.string()),
        pa.field("disease_activity", pa.string()),
        pa.field("umap_1", pa.float32()),
        pa.field("umap_2", pa.float32()),
        pa.field("distance_to_healthy", pa.float32()),
        *_emb_fields(n_gf),
    ]
)
cols_d_gf = {
    "cell_id": adata_disease.obs_names.astype(str).tolist(),
    "cell_type": adata_disease.obs["cell_type"].astype(str).tolist(),
    "disease_activity": adata_disease.obs["disease_activity"].astype(str).tolist(),
    "umap_1": U_d_gf[:, 0],
    "umap_2": U_d_gf[:, 1],
    "distance_to_healthy": dist_gf,
}
for j in range(n_gf):
    cols_d_gf[f"embedding_{j}"] = X_dfull[:, j]

p_d_gf = out_root / "sle_csle" / "geneformer_projected.parquet"
rows_d_gf = _write_table(sch_d_gf, cols_d_gf, p_d_gf)
key_d_gf = f"v{VERSION}/sle_csle/geneformer_projected.parquet"
_upload(p_d_gf, key_d_gf)

sch_d_gp = pa.schema(
    [
        pa.field("cell_id", pa.string()),
        pa.field("cell_type", pa.string()),
        pa.field("disease_activity", pa.string()),
        pa.field("umap_1", pa.float32()),
        pa.field("umap_2", pa.float32()),
        pa.field("distance_to_healthy", pa.float32()),
        *_emb_fields(n_gp),
    ]
)
cols_d_gp = {
    "cell_id": adata_disease.obs_names.astype(str).tolist(),
    "cell_type": adata_disease.obs["cell_type"].astype(str).tolist(),
    "disease_activity": adata_disease.obs["disease_activity"].astype(str).tolist(),
    "umap_1": U_d_gp[:, 0],
    "umap_2": U_d_gp[:, 1],
    "distance_to_healthy": dist_gp,
}
for j in range(n_gp):
    cols_d_gp[f"embedding_{j}"] = X_dgpt[:, j]

p_d_gp = out_root / "sle_csle" / "genept_projected.parquet"
rows_d_gp = _write_table(sch_d_gp, cols_d_gp, p_d_gp)
key_d_gp = f"v{VERSION}/sle_csle/genept_projected.parquet"
_upload(p_d_gp, key_d_gp)

sch_dist = pa.schema(
    [
        pa.field("cell_id", pa.string()),
        pa.field("cell_type", pa.string()),
        pa.field("disease_activity", pa.string()),
        pa.field("distance_geneformer", pa.float32()),
        pa.field("distance_genept", pa.float32()),
    ]
)
cols_dist = {
    "cell_id": adata_disease.obs_names.astype(str).tolist(),
    "cell_type": adata_disease.obs["cell_type"].astype(str).tolist(),
    "disease_activity": adata_disease.obs["disease_activity"].astype(str).tolist(),
    "distance_geneformer": dist_gf,
    "distance_genept": dist_gp,
}
p_dist = out_root / "sle_csle" / "distance_scores.parquet"
rows_dist = _write_table(sch_dist, cols_dist, p_dist)
key_dist = f"v{VERSION}/sle_csle/distance_scores.parquet"
_upload(p_dist, key_dist)

sch_dis = pa.schema(
    [
        pa.field("cell_id", pa.string()),
        pa.field("cell_type", pa.string()),
        pa.field("disease_activity", pa.string()),
        pa.field("distance_geneformer", pa.float32()),
        pa.field("distance_genept", pa.float32()),
        pa.field("disagreement", pa.float32()),
    ]
)
cols_dis = {
    **cols_dist,
    "disagreement": disag,
}
p_dis = out_root / "sle_csle" / "cross_model_disagreement.parquet"
rows_dis = _write_table(sch_dis, cols_dis, p_dis)
key_dis = f"v{VERSION}/sle_csle/cross_model_disagreement.parquet"
_upload(p_dis, key_dis)

manifest = [
    ("pbmc3k/geneformer_embeddings.parquet", p_h_gf, rows_h_gf, key_h_gf),
    ("pbmc3k/genept_embeddings.parquet", p_h_gp, rows_h_gp, key_h_gp),
    ("sle_csle/geneformer_projected.parquet", p_d_gf, rows_d_gf, key_d_gf),
    ("sle_csle/genept_projected.parquet", p_d_gp, rows_d_gp, key_d_gp),
    ("sle_csle/distance_scores.parquet", p_dist, rows_dist, key_dist),
    ("sle_csle/cross_model_disagreement.parquet", p_dis, rows_dis, key_dis),
]

print(json.dumps({"artifacts": [{"path": m[0], "rows": m[2], "bytes": m[1].stat().st_size, "s3_key": m[3]} for m in manifest]}, indent=2))


## Step 8: Provenance

Each `precompute_runs` row ties a model run to a **dataset FK** (`sle_csle`), **hyperparameters**, the **primary parquet S3 key**, and the **git SHA** captured at the top of the notebook. This is what makes the dashboard reproducible under ADR-002 / ADR-003.


In [ ]:
# Insert two precompute_runs rows (Geneformer + GenePT) via SQLModel async session.
import asyncio
import os
import ssl
import sys
from pathlib import Path
from urllib.parse import parse_qs, urlencode, urlparse, urlunparse

import nest_asyncio
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine
from sqlmodel import select
from sqlmodel.ext.asyncio.session import AsyncSession

nest_asyncio.apply()

_backend_candidates = [
    Path(os.environ.get("BACKEND_ROOT", "")).expanduser(),
    Path.cwd(),
    Path.cwd() / "backend",
    Path("/content/helical-bio-explorer/backend"),
    Path("/content/drive/MyDrive/helical-bio-explorer/backend"),
]
_backend_root = next(
    (p for p in _backend_candidates if p.parts and (p / "app" / "db" / "models.py").is_file()),
    None,
)
if _backend_root is None:
    raise RuntimeError(
        "Cannot import backend models. Clone the repo in Colab and set BACKEND_ROOT to the backend/ folder, "
        "or run this notebook from a checkout where backend/app exists."
    )
sys.path.insert(0, str(_backend_root))

from app.db.models import Dataset, PrecomputeRun


def _prepare_asyncpg_url(raw_url: str) -> tuple[str, dict[str, object]]:
    connect_args: dict[str, object] = {"statement_cache_size": 0}
    parsed = urlparse(raw_url)
    params = parse_qs(parsed.query, keep_blank_values=True)
    sslmode = params.pop("sslmode", [None])[0]
    if sslmode in ("require", "verify-ca", "verify-full"):
        connect_args["ssl"] = ssl.create_default_context()
    params.pop("channel_binding", None)
    clean_query = urlencode({k: v[0] for k, v in params.items()}, doseq=False)
    clean_url = urlunparse(parsed._replace(query=clean_query))
    return clean_url, connect_args


if not DATABASE_URL.startswith("postgresql+asyncpg://"):
    raise RuntimeError("DATABASE_URL must use postgresql+asyncpg:// for this notebook.")

_git = GIT_SHA if len(GIT_SHA) >= 7 else "0000000"


async def _insert_runs() -> None:
    clean_url, connect_args = _prepare_asyncpg_url(DATABASE_URL)
    engine = create_async_engine(clean_url, echo=False, connect_args=connect_args)
    session_maker = async_sessionmaker(engine, class_=AsyncSession, expire_on_commit=False)
    async with session_maker() as session:
        res = await session.exec(select(Dataset).where(Dataset.slug == "sle_csle"))
        ds = res.first()
        if ds is None or ds.id is None:
            raise RuntimeError("Dataset sle_csle not found. Run backend seed_sle_dataset.")

        session.add(
            PrecomputeRun(
                dataset_id=ds.id,
                model_name="geneformer",
                model_version="v1",
                parameters={"embedding_dim": int(n_gf), "max_input_genes": 2048},
                output_parquet_key=key_d_gf,
                git_sha=_git[:40],
            )
        )
        session.add(
            PrecomputeRun(
                dataset_id=ds.id,
                model_name="genept",
                model_version="v1",
                parameters={"embedding_dim": int(n_gp), "max_input_genes": 4096},
                output_parquet_key=key_d_gp,
                git_sha=_git[:40],
            )
        )
        await session.commit()


await _insert_runs()


## Verification

Independently re-read one parquet object from S3, assert row counts, print recent `precompute_runs` rows, and sum on-disk bytes for all six exports.


In [ ]:
import io

import boto3
import pyarrow.parquet as pq
from sqlmodel import select

from app.db.models import PrecomputeRun

s3_verify = boto3.client("s3", region_name=S3_REGION)
obj = s3_verify.get_object(Bucket=S3_BUCKET, Key=key_d_gf)
buf = io.BytesIO(obj["Body"].read())
rtab = pq.read_table(buf)
assert rtab.num_rows == int(rows_d_gf), (rtab.num_rows, rows_d_gf)
print(f"Verified S3 parquet rows for {key_d_gf}: {rtab.num_rows}")


async def _print_recent_runs() -> None:
    clean_url, connect_args = _prepare_asyncpg_url(DATABASE_URL)
    engine = create_async_engine(clean_url, echo=False, connect_args=connect_args)
    session_maker = async_sessionmaker(engine, class_=AsyncSession, expire_on_commit=False)
    async with session_maker() as session:
        res = await session.exec(select(PrecomputeRun).order_by(PrecomputeRun.created_at.desc()).limit(8))
        rows = res.all()
        for row in rows:
            print(row)


await _print_recent_runs()

_total_bytes = sum(m[1].stat().st_size for m in manifest)
print(f"Total local parquet bytes (6 files): {_total_bytes}")
print("Pipeline complete.")
